# DistilBERT v2 — Overnight Run (Kaggle)

Jalankan dengan **Save & Run All (Commit)** — berjalan di server Kaggle, PC boleh dimatikan total.

**Setup wajib sebelum run:**
1. Settings → Accelerator → **GPU T4 x2** (atau P100)
2. Settings → Internet → **ON**
3. Klik **Save Version** → pilih **Save & Run All (Commit)** → Save

Output tersimpan permanen di `/kaggle/working/` dan bisa diunduh besok dari tab **Output**.

Estimasi: 60-90 menit (full 200k, 3 epoch, fp16).

In [ ]:
# 0. PREFLIGHT — gagal cepat kalau environment salah (5 detik, bukan 24 menit)
import socket, sys, torch

ok = True
try:
    socket.create_connection(('github.com', 443), timeout=5).close()
    print('OK   Internet: ON')
except Exception as e:
    print('FAIL Internet: OFF ->', e)
    ok = False

if torch.cuda.is_available():
    print('OK   GPU:', torch.cuda.get_device_name(0))
else:
    print('FAIL GPU: tidak ada (CPU only)')
    ok = False

if not ok:
    print('\n' + '='*60)
    print('STOP. Jangan lanjut.')
    print('Sidebar kanan -> Session options:')
    print('  - Accelerator: GPU T4 x2')
    print('  - Internet: ON')
    print('Lalu Save Version -> Save & Run All (Commit)')
    print('='*60)
    sys.exit(1)

print('\nEnvironment siap, lanjut training.')

In [ ]:
# 1. Setup
!pip install -q transformers==5.20.2 datasets==5.1.0 accelerate scikit-learn==1.9.1

import torch, os
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Semua artefak ke /kaggle/working -> otomatis tersimpan sebagai Output notebook
WORK_DIR = '/kaggle/working'
os.makedirs(WORK_DIR, exist_ok=True)
print('Output dir:', WORK_DIR)

In [ ]:
# 2. Load full dataset + mapping
import random, numpy as np, torch
from datasets import load_dataset
from transformers import set_seed

SEED = 42
set_seed(SEED)

ID2LABEL = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

def map_label(l):
    return 0 if l in (0,1) else (1 if l==2 else 2)

ds = load_dataset('SetFit/amazon_reviews_multi_en')
train_full = ds['train'].map(lambda b: {'sentiment':[map_label(l) for l in b['label']]}, batched=True)
val_full   = ds['validation'].map(lambda b: {'sentiment':[map_label(l) for l in b['label']]}, batched=True)

from collections import Counter
print('Train mapped dist:', Counter(train_full['sentiment']))
print('(0=NEG, 1=NEU, 2=POS)')

In [ ]:
# 3. Class weight — NEUTRAL dinaikkan 2x
counts = np.bincount(train_full['sentiment'])
n = counts.sum()
base = n / (3 * counts)
boost = np.array([1.0, 2.0, 1.0])
class_weights = torch.tensor(base * boost, dtype=torch.float)
print('class_weights [NEG, NEU, POS]:', class_weights.tolist())

In [ ]:
# 4. Tokenisasi (max_len 160)
from transformers import AutoTokenizer
MAX_LEN = 160
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tok(b):
    return tokenizer(b['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

keep = lambda d: [c for c in d.column_names if c != 'sentiment']
train_tok = train_full.map(tok, batched=True, remove_columns=keep(train_full)).rename_column('sentiment','labels')
val_tok   = val_full.map(tok, batched=True, remove_columns=keep(val_full)).rename_column('sentiment','labels')
train_tok.set_format('torch'); val_tok.set_format('torch')
print('Tokenized:', len(train_tok), 'train /', len(val_tok), 'val')

In [ ]:
# 5. Model + Weighted Trainer
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        cw = class_weights.to(outputs.logits.device)
        loss = nn.functional.cross_entropy(outputs.logits, labels, weight=cw)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'f1_macro': f1_score(p.label_ids, preds, average='macro')}

fp16 = torch.cuda.is_available()
args = TrainingArguments(
    output_dir=WORK_DIR + '/checkpoint_v2',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=fp16,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    seed=SEED,
    data_seed=SEED,
    report_to=[],
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_tok,
                          eval_dataset=val_tok, processing_class=tokenizer,
                          data_collator=DataCollatorWithPadding(tokenizer),
                          compute_metrics=compute_metrics)

print(f'Mulai training: full 200k, 3 epoch, fp16={fp16}')
trainer.train()
print('Training selesai')

In [ ]:
# 6. Evaluasi validation + F1 per kelas
val_metrics = trainer.evaluate()
print('=== Validation Metrics (v2) ===')
for k,v in val_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v,float) else f'  {k}: {v}')

preds = trainer.predict(val_tok)
y_pred = np.argmax(preds.predictions, axis=1)
f1s = f1_score(preds.label_ids, y_pred, average=None, labels=[0,1,2])
print('\nF1 per kelas:')
for i,name in ID2LABEL.items():
    print(f'  {name}: {f1s[i]:.4f}')
print('\nBandingkan v1: NEUTRAL 0.403 | target v2: NEUTRAL > 0.50')

In [ ]:
# 7. Simpan model + metrics ke /kaggle/working (otomatis jadi Output)
import json, shutil
model.save_pretrained(WORK_DIR + '/final_v2')
tokenizer.save_pretrained(WORK_DIR + '/final_v2')

# simpan metrics validation sebagai file
with open(WORK_DIR + '/metrics_v2_validation.json','w') as f:
    json.dump({**{k: float(v) for k,v in val_metrics.items() if isinstance(v,(int,float))},
               'f1_per_class': {ID2LABEL[i]: float(f1s[i]) for i in range(3)}}, f, indent=2)

!cd /kaggle/working && zip -r model_v2.zip final_v2/ > /dev/null
!ls -lh /kaggle/working
print('\n=== SELESAI ===')
print('Besok: tab Output notebook -> download model_v2.zip + metrics_v2_validation.json')
print('Bandingkan accuracy v2 vs v1 (0.75). Kalau v2 lebih tinggi, ganti model/final.')
print('Kalau v2 lebih rendah, keep v1 — tidak ada yang perlu diubah.')